# FEM solver for Darcy's problem
Need a benchmark to compare with FNO
Useful notebook: https://github.com/jorgensd/dolfinx-tutorial/blob/main/chapter2/nonlinpoisson_code.ipynb

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as tri
from mpi4py import MPI
from dolfinx import mesh, fem
from dolfinx.fem.petsc import LinearProblem
import ufl


In [ ]:
# ── 1. MESH ──────────────────────────────────────────────────────────────────
# 16x16 grid to match your Neural Operator parametrisation
Nx, Ny = 16, 16
domain = mesh.create_unit_square(MPI.COMM_WORLD, Nx, Ny)

In [ ]:
# ── 2. PERMEABILITY FIELD a(x) ───────────────────────────────────────────────
# Piecewise-constant per cell (DG0): one value per triangle
# With a 16x16 quad mesh you get 16*16*2 = 512 triangles
V_k = fem.functionspace(domain, ("DG", 0))
a_func = fem.Function(V_k)

# Replace this with your actual a(x) sample:
rng = np.random.default_rng(0)
a_func.x.array[:] = np.exp(rng.normal(0, 1, a_func.x.array.shape))

In [ ]:
# ── 3. FUNCTION SPACE & BCs ──────────────────────────────────────────────────
V = fem.functionspace(domain, ("Lagrange", 1))

# u = 0 on ALL of ∂Ω (homogeneous Dirichlet)
fdim = domain.topology.dim - 1
all_boundary_facets = mesh.locate_entities_boundary(
    domain, fdim, lambda x: np.full(x.shape[1], True, dtype=bool)
)
bc = fem.dirichletbc(
    fem.Constant(domain, 0.0),
    fem.locate_dofs_topological(V, fdim, all_boundary_facets),
    V
)

In [ ]:
# ── 4. VARIATIONAL FORM ──────────────────────────────────────────────────────
# PDE: -∇·(a(x)∇u) = 1  →  ∫ a(x) ∇u·∇v dx = ∫ v dx
u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

a_form = a_func * ufl.dot(ufl.grad(u), ufl.grad(v)) * ufl.dx
L_form = v * ufl.dx     # RHS = 1 everywhere

In [ ]:
# ── 5. SOLVE ─────────────────────────────────────────────────────────────────
problem = LinearProblem(a_form, L_form, bcs=[bc], petsc_options={
    "ksp_type": "cg",
    "pc_type": "hypre",
    "pc_hypre_type": "boomeramg"
})
uh = problem.solve()

In [ ]:
# ── 6. EXTRACT SOLUTION ON 16x16 GRID ───────────────────────────────────────
# DOF coordinates for P1 space → (Nx+1)*(Ny+1) = 17*17 = 289 points
coords = V.tabulate_dof_coordinates()[:, :2]
u_vals = uh.x.array.real

# Interpolate onto a regular 16x16 grid (matching your NO output grid)
from scipy.interpolate import griddata
grid_x, grid_y = np.meshgrid(
    np.linspace(0, 1, Nx), np.linspace(0, 1, Ny)
)
u_grid = griddata(coords, u_vals, (grid_x, grid_y), method="linear")

In [ ]:
# ── 7. PLOT ───────────────────────────────────────────────────────────────────
# Build triangulation for the FEM mesh plots
domain.topology.create_connectivity(2, 0)
cells = domain.topology.connectivity(2, 0).array.reshape(-1, 3)
triangulation = tri.Triangulation(coords[:, 0], coords[:, 1], cells)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Panel 1 — input: permeability field a(x) on 16x16 grid
a_grid = griddata(
    coords,
    # a is DG0 (cell-based), so sample it at cell centres
    # easier: just show the 16x16 raw input
    np.log(a_func.x.array.real),   # log scale is conventional
    (grid_x, grid_y), method="nearest"
)
im0 = axes[0].imshow(
    a_grid, origin="lower", extent=[0,1,0,1], cmap="viridis"
)
plt.colorbar(im0, ax=axes[0])
axes[0].set_title("Input: log a(x)")
axes[0].set_aspect("equal")

# Panel 2 — FEM solution on FEM mesh (high fidelity view)
tc = axes[1].tricontourf(triangulation, u_vals, levels=50, cmap="inferno")
plt.colorbar(tc, ax=axes[1])
axes[1].set_title("FEM solution u(x)")
axes[1].set_aspect("equal")

# Panel 3 — FEM solution resampled on 16x16 grid (apples-to-apples with NO)
im2 = axes[2].imshow(
    u_grid, origin="lower", extent=[0,1,0,1], cmap="inferno"
)
plt.colorbar(im2, ax=axes[2])
axes[2].set_title("FEM on 16×16 grid")
axes[2].set_aspect("equal")

plt.tight_layout()
plt.savefig("darcy_fem.png", dpi=150)
plt.show()